# Face Recognition with Neural Networks: From Fully-Connected to Convolutional

The **Labeled Faces in the Wild (LFW)** dataset contains real-world face photographs collected from the web, covering hundreds of individuals under varying lighting, pose, and expression conditions. The task is a **multi-class classification** problem: given a 62×47 grayscale image of a face, predict whose face it is. This is a canonical benchmark for evaluating face recognition systems in unconstrained settings.

In this notebook we build and compare two PyTorch architectures — a **fully-connected (MLP) network** that treats each image as a flat pixel vector, and a **convolutional neural network (CNN)** that exploits the spatial structure of images. By training both models on the same data and evaluating them on a held-out test set, we will see how inductive biases baked into the architecture affect both accuracy and parameter efficiency.

## 1. Background

### The LFW Dataset

LFW was introduced by Huang et al. (2007) and has since become a standard benchmark in face recognition research. Images are scraped from news articles and web pages, so they reflect real-world diversity in:
- **Pose**: faces can be tilted, turned, or seen in profile.
- **Illumination**: lighting varies from studio quality to harsh shadows.
- **Expression**: neutral, smiling, surprised, and many other expressions appear.
- **Occlusion**: glasses, scarves, microphones, and other objects may partially cover faces.

We use the `sklearn` wrapper, which crops and resizes each image to **62×47 pixels** (height × width) and provides integer class labels aligned with a list of names.

### Why Face Recognition Is Hard

The within-class variance (different images of the same person) can be larger than the between-class variance (images of different people) when conditions vary widely. A successful model must learn an identity-discriminative representation that is simultaneously **invariant** to nuisance factors like pose and lighting.

### Two Approaches

| | MLP | CNN |
|---|---|---|
| Input representation | Flat vector of $62 \times 47 = 2914$ pixels | 2-D grid, shape $(1, 62, 47)$ |
| Spatial awareness | None — pixel order is arbitrary to the model | Yes — local filters slide over the image |
| Parameter sharing | No | Yes — same filter reused at every position |
| Typical strength | Simple, fast, works on small data | Extracts hierarchical spatial features |

Both models will be trained with **Adam** and **cross-entropy loss**, the standard recipe for multi-class classification.

## 2. Imports

## 3. Dataset Loading & Exploration

We fetch LFW via scikit-learn, keeping only identities that have **at least 70 images**. This threshold filters out individuals with too few examples, which would make learning their identity very difficult and would create severe class imbalance. The `data` attribute contains images already flattened to shape `(n_samples, 2914)`, while `images` retains the 2-D shape `(n_samples, 62, 47)` — useful for visualization.

The grid above illustrates the diversity in lighting and pose even within a single identity. Notice that some images are brighter, some darker, and faces are not perfectly centred — all realistic challenges.

## 4. Data Preprocessing

We split the data 80 / 20 into training and test sets (stratification is omitted here for simplicity). The pixel values returned by `sklearn` are already 32-bit floats in the range $[0, 255]$; we convert them directly to tensors without further normalization, matching the original setup.

Labels are **integer indices** into `target_names`. PyTorch's `nn.CrossEntropyLoss` expects raw class indices (not one-hot vectors) and internally applies the log-softmax, so no additional label encoding is required:

$$\mathcal{L}_{\text{CE}} = -\log \frac{e^{z_{y_i}}}{\sum_{k} e^{z_k}}$$

where $z_k$ is the $k$-th logit output by the network and $y_i$ is the true class index.

## 5. DataLoaders

We wrap the tensors in a `TensorDataset` and then a `DataLoader`. Mini-batch stochastic gradient descent (here with batch size 16) gives a good trade-off between gradient noise (which can help escape local minima) and computational efficiency.

## 6. Model 1: Fully-Connected Network

The simplest baseline is a **multi-layer perceptron (MLP)**: we treat the image as a flat vector of $62 \times 47 = 2914$ scalar features and pass it through a stack of linear layers with ReLU non-linearities.

Each neuron in the first hidden layer computes a weighted sum of **all** input pixels:

$$z_j = \sum_{i=1}^{2914} w_{ji}\, x_i + b_j, \qquad a_j = \text{ReLU}(z_j) = \max(0, z_j)$$

By flattening the image we **discard all spatial relationships**: pixels at position $(0,0)$ and $(61,46)$ are treated identically regardless of proximity. The model must re-learn spatial correlations from scratch, purely from labelled examples — an information-theoretically wasteful approach compared to architectures that encode spatial priors.

Architecture: $2914 \to 128 \to 64 \to n_{\text{classes}}$

## 7. Training Model 1

We train for 15 epochs using the **Adam** optimizer (an adaptive learning-rate method that combines momentum and RMSProp) and `CrossEntropyLoss`. After training we evaluate accuracy on the held-out test set.

## 8. Model 2: Convolutional Neural Network

### How Convolution Works

A **convolutional layer** applies a small learnable filter (kernel) $k$ of size $m \times n$ across the image $f$ by computing the discrete cross-correlation at every spatial position:

$$(f * k)[i,j] = \sum_{m}\sum_{n} f[i+m,\, j+n] \cdot k[m,n]$$

Key properties:
- **Local connectivity**: each output neuron depends only on a small receptive field, not the entire image.
- **Parameter sharing**: the same filter weights $k$ are applied at every position, drastically reducing the number of parameters compared to a fully-connected layer.
- **Translation equivariance**: if an edge moves in the input, the corresponding activation moves in the output — exactly the right inductive bias for images.

### MaxPooling

After activation, a **max-pooling** layer partitions the feature map into non-overlapping windows and retains the maximum value in each:

$$p[i,j] = \max_{(r,s)\in\text{window}} a[i \cdot s + r,\; j \cdot s + s]$$

This achieves two goals: (1) it reduces the spatial dimensions, decreasing computation in subsequent layers, and (2) it provides a small degree of **translation invariance** — a feature detected slightly off-centre still produces a strong pooled activation.

### Architecture

$$\underbrace{(1,62,47)}_{\text{input}} \xrightarrow{\text{Conv}(5\times5,16)} \xrightarrow{\text{ReLU}} \xrightarrow{\text{MaxPool}(2)} \xrightarrow{\text{Flatten}} \xrightarrow{\text{Linear}} \underbrace{n_{\text{classes}}}_{\text{logits}}$$

## 9. Training Model 2

### Discussion

On a dataset of this size (~1000 training images) it is common to observe the CNN matching or slightly outperforming the MLP, though the gap may be modest. The CNN's advantage grows with dataset size and image resolution because:
- More data allows the filters to specialise into meaningful detectors (edges, corners, textures).
- Higher resolution images contain richer spatial structure that flat vectors cannot exploit.

On small, low-resolution datasets the MLP's simplicity can be a virtue — it is less prone to overfitting induced by an overly expressive feature hierarchy.

## 10. CNN Feature Visualization

One of the most illuminating properties of CNNs is that we can inspect the **intermediate feature maps** — the output of each layer for a given input image. This lets us see what information the network has extracted at each stage.

We will visualize:
1. The **raw image** (62×47 grayscale).
2. The **Conv2D output** (before ReLU) for the first 8 filters — each map highlights a different local pattern.
3. The **MaxPool output** for the same 8 filters — spatially compressed by a factor of 2 in each dimension.

### What Are We Seeing?

Each **Conv2D feature map** responds most strongly where the learned $5 \times 5$ filter pattern matches the local image content. Different filters specialize in different orientations of edges, areas of high contrast, or smooth gradients — even though we never told the network to look for edges; it discovered these patterns automatically through gradient descent.

The **MaxPool feature maps** are spatially smaller (each dimension is halved) but retain the peak activations. Notice that the overall structure of the face is still recognizable in the stronger feature maps, confirming that the network preserves identity-relevant information while discarding fine-grained positional noise.

Deeper CNNs stack many such layers, progressively combining low-level features (edges) into mid-level features (eye corners, nose bridge) and ultimately into high-level representations (face identity).

## 11. Results Summary

Let us compare both models quantitatively.

### Discussion

**Parameter count.** The MLP has $2914 \times 128 + 128 + 128 \times 64 + 64 + 64 \times n_{\text{classes}} + n_{\text{classes}}$ parameters — dominated by the first layer, which must maintain a separate weight for every pixel-to-neuron pair. The CNN instead uses $16$ filters of size $5 \times 5 \times 1 = 25$ weights each (plus bias), totalling only $16 \times 26 = 416$ parameters in the convolutional layer. The large linear classifier on top of the flattened feature maps makes up most of the CNN's total, but it operates on a much smaller spatial footprint.

**When CNNs win.** CNNs are the dominant architecture for image tasks at scale because:
- Convolutional filters encode **translation equivariance** for free — the same face detector works anywhere in the image.
- Hierarchical feature extraction produces compact, generalizable representations.
- Parameter sharing makes CNNs dramatically more sample-efficient.

**When MLPs are competitive.** On very small datasets with low-resolution images (like this LFW subset), the spatial hierarchy a CNN aims to build may not have enough data to train properly. In such regimes a simple MLP can be comparably accurate, and is easier to reason about and tune.

**Next steps.** To push accuracy further on LFW one would typically:
- Use **deeper CNN architectures** (ResNet, VGG) pre-trained on large face datasets (transfer learning).
- Apply **data augmentation** (random horizontal flips, small rotations) to artificially expand the training set.
- Add **batch normalization** and **dropout** to regularize training.
- Use **metric learning** objectives (e.g., triplet loss) instead of cross-entropy to learn more discriminative embeddings.